# 1. Data audit
NullPointer / CSE558. Run the full chunked CSV analysis with `python scripts/run_all.py --data PATH` first. This notebook inspects its actual saved audit without loading millions of raw rows into notebook output.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if not (root / 'outputs').exists(): root = root.parent
assert (root / 'outputs/results.json').exists(), 'Run scripts/run_all.py first'
def read(name): return json.loads((root / 'outputs' / name).read_text(encoding='utf-8'))


In [2]:
a=read('dataset_audit.json')
print({k:a[k] for k in ['rows','columns','file_bytes','file_sha256','duplicate_rows','duplicate_ids','issue_date_min','issue_date_max']})
assert sum(a['status_counts'].values()) == a['rows']
print(pd.Series(a['status_counts']).to_string())

{'rows': 2260701, 'columns': 151, 'file_bytes': 1675133810, 'file_sha256': '3eae03c28fd9d2e8a076ebeb73507e8d4d0f44d90500decdb0936e0933d1f36a', 'duplicate_rows': 0, 'duplicate_ids': 0, 'issue_date_min': '2007-06-01 00:00:00', 'issue_date_max': '2018-12-01 00:00:00'}
Fully Paid                                             1076751
Charged Off                                             268559
Current                                                 878317
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Default                                                     40
<missing>                                                   33
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761


## Profile discrepancy
The profile was not a substitute for the full CSV. Its 314,168 rows and 162 columns differ from the actual audit. Derived dates may explain its column inventory, but that profiling run is not independently reconstructible.

In [3]:
print(pd.read_csv(root/'outputs/tables/profile_comparison.csv').to_string(index=False))
m=pd.read_csv(root/'outputs/tables/missingness.csv')
assert len(m)==a['columns']
print(m.sort_values('missing_count',ascending=False).head(20).to_string(index=False))

           metric  profile  actual
             rows   314168 2260701
          columns      162     151
   duplicate_rows        0       0
       unique_ids   314168 2260701
rows_with_missing   314168 2260701
                                    column  missing_count  missing_percent inferred_type
                                 member_id        2260701       100.000000   all_missing
orig_projected_additional_accrued_interest        2252050        99.617331       numeric
                           hardship_reason        2249784        99.517097          text
            hardship_payoff_balance_amount        2249784        99.517097       numeric
              hardship_last_payment_amount        2249784        99.517097       numeric
                   payment_plan_start_date        2249784        99.517097          text
                             hardship_type        2249784        99.517097          text
                           hardship_status        2249784        99.517097    

## Interpretation
Missingness may be structural for joint applications and hardship/settlement programs. ID uniqueness is not borrower independence. The raw file is read-only; source size/mtime are checked and its SHA-256 is recorded. See the full feature audit before using variables as origination predictors.